In [ ]:
import re
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches


In [ ]:
LOG_DIR = Path("/home/gabriel/Projects/Anchoring_Study/anch_frac_0.5/Results/lc_length_6/Log")
DT = 0.005

PATTERN = re.compile(
    r"phase_separation_(nematic|smectic)_lc_frac_(\d+\.\d+)_anch_str_([-\d.]+)_Nb_\d+_rep_(\d+)_Nb_\d+\.dat"
)


In [ ]:
tasks = []
for path in sorted(LOG_DIR.glob("*.dat")):
    m = PATTERN.match(path.name)
    if m and m.group(3) == "0.0":
        tasks.append((str(path), m.group(1), float(m.group(2)), int(m.group(4)), DT))

print(f"Found {len(tasks)} files:")
for t in tasks:
    print(f"  {Path(t[0]).name}  →  phase={t[1]}, lc_frac={t[2]}, rep={t[3]}")


In [ ]:
def _load_log(task):
    path, phase, lc_frac, rep, dt = task
    with open(path) as f:
        header = f.readline().lstrip("# ").strip().split()
    df = pd.read_csv(path, comment="#", sep=r"\s+", header=None, names=header)
    df["time"] = df["step"] * dt
    df["phase"] = phase
    df["lc_frac"] = lc_frac
    df["rep"] = rep
    return df[["time", "LambdaMaierSaupe0", "phase", "lc_frac", "rep"]].rename(
        columns={"LambdaMaierSaupe0": "lambda"}
    )


def _choose_n_workers(n):
    if n % 4 == 0:
        return 4
    elif n % 3 == 0:
        return 3
    return 2


n_workers = _choose_n_workers(len(tasks))
print(f"Loading {len(tasks)} files with {n_workers} workers...")
with Pool(n_workers) as pool:
    dfs = pool.map(_load_log, tasks)
all_data = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(all_data):,} rows total.")
all_data.head()


In [ ]:
cmap = plt.cm.tab10.colors
lc_fracs = sorted(all_data["lc_frac"].unique())
phases = ["nematic", "smectic"]
phase_titles = {"nematic": r"Nematic ($\tilde{\mu}=25$)", "smectic": r"Smectic ($\tilde{\mu}=50$)"}
frac_colors = {f: cmap[i] for i, f in enumerate(lc_fracs)}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, phase in zip(axes, phases):
    for frac in lc_fracs:
        grp = all_data[(all_data["phase"] == phase) & (all_data["lc_frac"] == frac)]
        stats = grp.groupby("time")["lambda"].agg(["mean", "std"]).reset_index()
        color = frac_colors[frac]
        ax.plot(stats["time"], stats["mean"], color=color, lw=1.8,
                label=f"$f_{{\\rm LC}}={frac}$")
        ax.fill_between(
            stats["time"],
            stats["mean"] - stats["std"],
            stats["mean"] + stats["std"],
            color=color, alpha=0.2,
        )
    ax.set_title(phase_titles[phase], fontsize=13)
    ax.set_xlabel("Time (τ)", fontsize=12)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=11)

axes[0].set_ylabel("λ (Nematic Order Parameter)", fontsize=12)

plt.tight_layout()
plt.savefig("order_parameter_vs_time.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
DENSITY_BASE = Path("/home/gabriel/Projects/Anchoring_Study/anch_frac_0.5/Results/lc_length_6/Density")
MAX_STEPS    = 2_000_000
N_DEN_FRAMES = 35
FREQ0        = 200
LC_TYPE      = 1  # rho1 = liquid crystal density


def _log_frame_steps():
    ratio = (MAX_STEPS / FREQ0) ** (1.0 / (N_DEN_FRAMES - 1))
    freq  = float(FREQ0)
    steps = []
    for _ in range(N_DEN_FRAMES):
        steps.append(round(freq))
        freq *= ratio
    return np.array(steps, dtype=int)


FRAME_STEPS = _log_frame_steps()
FRAME_TIMES = FRAME_STEPS * DT


def _read_density_bin(path):
    with open(path, "rb") as f:
        Dim    = np.frombuffer(f.read(4),     dtype=np.int32)[0]
        Nx     = np.frombuffer(f.read(4*Dim), dtype=np.int32).copy()
        L      = np.frombuffer(f.read(4*Dim), dtype=np.float32).copy()
        ntypes = np.frombuffer(f.read(4),     dtype=np.int32)[0]
        raw    = np.frombuffer(f.read(),       dtype=np.float32)
    M = int(np.prod(Nx))
    return Nx, L, raw.reshape(-1, ntypes, M)


def _sk_peak(rho_lc_flat, Nx, L):
    """Radially averaged S(q) peak q* for the LC density field."""
    shape = (int(Nx[2]), int(Nx[1]), int(Nx[0]))  # iz-slowest for correct FFT
    rho   = rho_lc_flat.reshape(shape)
    k_1d  = [2*np.pi * np.fft.fftfreq(shape[d], d=float(L[2-d])/shape[d]) for d in range(3)]
    kz, ky, kx = np.meshgrid(k_1d[0], k_1d[1], k_1d[2], indexing='ij')
    k_mag = np.sqrt(kx**2 + ky**2 + kz**2)
    dk    = 2*np.pi / float(L[0])
    k_bins    = np.arange(0, k_mag.max() + dk, dk)
    k_centers = 0.5 * (k_bins[:-1] + k_bins[1:])

    sk      = (np.abs(np.fft.fftn(rho)) / rho.size) ** 2
    k_flat  = k_mag.ravel()
    sk_flat = sk.ravel()
    bidx    = np.digitize(k_flat, k_bins) - 1
    valid   = (bidx >= 0) & (bidx < len(k_centers)) & (k_flat > 0)
    sk_r = np.bincount(bidx[valid], weights=sk_flat[valid], minlength=len(k_centers)).astype(float)
    cnt  = np.bincount(bidx[valid],                          minlength=len(k_centers)).astype(float)
    sk_r[cnt > 0] /= cnt[cnt > 0]

    # Focus on phase-separation length scales (below smectic peak ~2pi/Nb)
    ps_band = (k_centers > 0) & (k_centers < 0.8)
    if ps_band.any() and sk_r[ps_band].max() > 0:
        q_star   = k_centers[ps_band][np.argmax(sk_r[ps_band])]
        R_domain = 2*np.pi / q_star
    else:
        q_star, R_domain = np.nan, np.nan
    return q_star, R_domain


def _analyze_density(task):
    path, phase, lc_frac, rep = task
    Nx, L, all_rho = _read_density_bin(path)
    records = []
    for i in range(all_rho.shape[0]):
        rho_lc   = all_rho[i, LC_TYPE, :]  # flat, ix-major
        rho_mean = rho_lc.mean()
        # Fraction of LC mass in above-mean voxels (→ 0.5 uniform, → 1.0 fully separated)
        f_conc   = float(rho_lc[rho_lc > rho_mean].sum() / rho_lc.sum()) if rho_lc.sum() > 0 else 0.5
        q_star, R_domain = _sk_peak(rho_lc, Nx, L)
        records.append(dict(frame=i, f_conc=f_conc, q_star=q_star, R_domain=R_domain,
                            phase=phase, lc_frac=lc_frac, rep=rep))
    return records


den_tasks = []
for phase in ["nematic", "smectic"]:
    for frac in [0.3, 0.5, 0.7]:
        for rep in [0, 1, 2]:
            p = (DENSITY_BASE / f"{phase}_lc_frac_{frac}_anch_str_0.0" /
                 f"phase_separation_{phase}_lc_frac_{frac}_anch_str_0.0_Nb_6_rep_{rep}_Nb_6_den.bin")
            if p.exists():
                den_tasks.append((str(p), phase, float(frac), rep))

n_w = _choose_n_workers(len(den_tasks))
print(f"Analyzing {len(den_tasks)} density files with {n_w} workers ...")
with Pool(n_w) as pool:
    batches = pool.map(_analyze_density, den_tasks)
den_records = [r for b in batches for r in b]
den_df = pd.DataFrame(den_records)
den_df["time"] = FRAME_TIMES[den_df["frame"].values]
print(f"{len(den_df)} frame-records loaded.")
den_df.head()


In [ ]:
# Interpolate lambda_global from log file at the density frame times
log_interp = []
for (phase, lc_frac, rep), grp in all_data.groupby(["phase", "lc_frac", "rep"]):
    grp_sorted = grp.sort_values("time")
    lam_at_frames = np.interp(FRAME_TIMES, grp_sorted["time"].values, grp_sorted["lambda"].values)
    for i, t in enumerate(FRAME_TIMES):
        log_interp.append(dict(phase=phase, lc_frac=lc_frac, rep=rep, frame=i,
                               time=t, lambda_global=lam_at_frames[i]))
log_interp_df = pd.DataFrame(log_interp)

# Merge with density analysis
local_df = den_df.merge(log_interp_df, on=["phase", "lc_frac", "rep", "frame", "time"])

# lambda_local ≈ lambda_global / f_conc
# f_conc: fraction of LC mass in LC-rich voxels (0.5 → uniform, 1.0 → fully separated)
# Note: this is a lower bound on true local order since inter-domain director
# cancellation also suppresses lambda_global.
local_df["lambda_local"] = local_df["lambda_global"] / local_df["f_conc"].clip(lower=0.5)

print(local_df[["phase", "lc_frac", "rep", "frame", "time",
                "f_conc", "R_domain", "lambda_global", "lambda_local"]].head(10))


In [ ]:
cmap       = plt.cm.tab10.colors
lc_fracs   = sorted(local_df["lc_frac"].unique())
phases     = ["nematic", "smectic"]
phase_titles = {"nematic": r"Nematic ($\tilde{\mu}=25$)",
                 "smectic": r"Smectic ($\tilde{\mu}=50$)"}
frac_colors  = {f: cmap[i] for i, f in enumerate(lc_fracs)}
phase_ls     = {"nematic": "-", "smectic": "--"}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: lambda_local vs time
ax = axes[0]
for phase in phases:
    for frac in lc_fracs:
        sub = local_df[(local_df["phase"] == phase) & (local_df["lc_frac"] == frac)]
        stats = sub.groupby("time")[["lambda_local"]].agg(["mean", "std"]).reset_index()
        stats.columns = ["time", "mean", "std"]
        color = frac_colors[frac]
        ax.plot(stats["time"], stats["mean"], color=color, ls=phase_ls[phase], lw=1.8)
        ax.fill_between(stats["time"],
                        stats["mean"] - stats["std"],
                        stats["mean"] + stats["std"],
                        color=color, alpha=0.2)
ax.set_xlabel("Time (\u03c4)", fontsize=12)
ax.set_ylabel("\u03bb\u2099\u2092\u02e3 (local order in LC domain)", fontsize=12)
ax.set_title("Local nematic order vs time", fontsize=12)
ax.set_xlim(left=0); ax.set_ylim(bottom=0)

# Panel 2: f_conc vs time
ax = axes[1]
for phase in phases:
    for frac in lc_fracs:
        sub = local_df[(local_df["phase"] == phase) & (local_df["lc_frac"] == frac)]
        stats = sub.groupby("time")[["f_conc"]].agg(["mean", "std"]).reset_index()
        stats.columns = ["time", "mean", "std"]
        color = frac_colors[frac]
        ax.plot(stats["time"], stats["mean"], color=color, ls=phase_ls[phase], lw=1.8)
        ax.fill_between(stats["time"],
                        stats["mean"] - stats["std"],
                        stats["mean"] + stats["std"],
                        color=color, alpha=0.2)
ax.axhline(0.5, color="grey", ls=":", lw=1, label="uniform (0.5)")
ax.set_xlabel("Time (\u03c4)", fontsize=12)
ax.set_ylabel("f\u209c\u2092\u2099\u2093 (LC mass fraction in rich domain)", fontsize=12)
ax.set_title("Phase separation extent vs time", fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(left=0); ax.set_ylim(0.4, 1.05)

# Panel 3: lambda_local vs R_domain (curvature plot)
ax = axes[2]
for phase in phases:
    for frac in lc_fracs:
        sub = local_df[(local_df["phase"] == phase) & (local_df["lc_frac"] == frac)].dropna(subset=["R_domain"])
        stats = sub.groupby("frame")[["R_domain", "lambda_local"]].mean().reset_index()
        color = frac_colors[frac]
        ax.plot(stats["R_domain"], stats["lambda_local"], color=color, ls=phase_ls[phase],
                lw=1.8, label=f"$f_{{\\rm LC}}={frac}$, {phase}")
        # Arrow indicating time direction (early → late = left → right)
        if len(stats) > 5:
            mid = len(stats) // 2
            ax.annotate("", xy=(stats["R_domain"].iloc[mid+1], stats["lambda_local"].iloc[mid+1]),
                        xytext=(stats["R_domain"].iloc[mid], stats["lambda_local"].iloc[mid]),
                        arrowprops=dict(arrowstyle="->", color=color, lw=1.2))
ax.set_xlabel("Domain size R (simulation units)", fontsize=12)
ax.set_ylabel("\u03bb\u2099\u2092\u02e3", fontsize=12)
ax.set_title("Local order vs domain size\n(arrow = increasing time)", fontsize=12)
ax.set_xlim(left=0); ax.set_ylim(bottom=0)

# Shared legend
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
frac_handles  = [mpatches.Patch(color=frac_colors[f], label=f"$f_{{\\rm LC}}={f}$") for f in lc_fracs]
phase_handles = [mlines.Line2D([], [], color="k", ls=phase_ls[p], lw=1.8, label=p.capitalize()) for p in phases]
fig.legend(handles=frac_handles + phase_handles, fontsize=10, loc="lower center",
           ncol=len(lc_fracs)+len(phases), bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.savefig("local_lambda_vs_time_and_curvature.png", dpi=150, bbox_inches="tight")
plt.show()
